In [1]:
import multixrank
import os
from multiprocessing import Pool
import pandas as pd
from pathlib import Path

In [2]:
def process_config(i):
    multixrank_obj = multixrank.Multixrank(config=f"mxr_config/config_{i}.yaml", wdir='./')
    ranking_df = multixrank_obj.random_walk_rank()
    os.makedirs(f"mxr_results/output_{i}", exist_ok=True)
    multixrank_obj.write_ranking(ranking_df, path=f"mxr_results/output_{i}/")

In [3]:
# with Pool(20) as pool:
#     pool.map(process_config, range(411, 412))

In [ ]:
results_dir = Path("mxr_results")
orphan_df = pd.read_csv("../knowledge_graph/orphan_associations.tsv", sep="\t")
unique_drugs = orphan_df["from"].unique()

ranks_per_drug = []
hits_at_10_per_drug = []

for i, drug_id in enumerate(unique_drugs, start=1):
    result_path = results_dir / f"output_{i}" / "multiplex_Disease.tsv"
    
    if not result_path.exists():
        print(f"Missing output for seed {i}")
        continue

    ranking_df = pd.read_csv(result_path, sep="\t")
    ranked_diseases = ranking_df["node"].tolist()

    target_diseases = orphan_df[orphan_df["from"] == drug_id]["to"].tolist()

    rr_list = []
    hit_count = 0

    for target in target_diseases:
        if target in ranked_diseases:
            rank = ranked_diseases.index(target) + 1
            rr_list.append(1 / rank)
            if rank <= 10:
                hit_count += 1

    if rr_list:
        mrr = sum(rr_list) / len(rr_list)
        ranks_per_drug.append(mrr)

        hit_at_10_ratio = hit_count / len(target_diseases)
        hits_at_10_per_drug.append(hit_at_10_ratio)

    else:
        print(f"Target disease not found for seed {i}")

if ranks_per_drug:
    global_mrr = sum(ranks_per_drug) / len(ranks_per_drug)
    mean_hit_at_10 = sum(hits_at_10_per_drug) / len(hits_at_10_per_drug)
    print(f"Results on {len(ranks_per_drug)} drugs :")
    print(f"MRR       : {global_mrr:.4f}")
    print(f"Hit@10    : {mean_hit_at_10:.4f}")
else:
    print("No target found in any run.")


\Results on 411 drugs :
MRR       : 0.0282
Hit@10    : 0.0932
